# Portfolio Re-Forecasting — Buckeye | PP | Beginning Outstanding Loans
---
**Benchmarks (test period 2025-01 → 2026-01):**

| Forecast | Test MAPE |
|---|---|
| `2025 0+12` (existing statistical) | **0.67%** ← *target to beat* |
| `PRED` (senior Prophet) | **1.38%** ← *baseline* |

**Key differences vs Fingerhut notebook:**
- Filters on both `PORTFOLIO = Buckeye` and `SUB_PORTFOLIO = PP`
- Test data available (2025-01 → 2026-01) → **real test MAPE computed for every model**
- N_FORECAST = 13 (Jan-2025 → Jan-2026)
- Every model produces the exact same style plot as the senior notebook (Actual + PRED + 2025 0+12 over the test window)


## 1. Installations

In [ ]:
%%capture
!pip install --proxy=192.168.2.10:8080 --upgrade snowflake-connector-python[pandas]
!pip install --proxy=192.168.2.10:8080 --upgrade keyring seaborn
!pip install --proxy=192.168.2.10:8080 prophet
!pip install --proxy=192.168.2.10:8080 hyperopt
!pip install --proxy=192.168.2.10:8080 pmdarima
!pip install --proxy=192.168.2.10:8080 xgboost
!pip install --proxy=192.168.2.10:8080 lightgbm


## 2. Imports

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import snowflake.connector
import datetime

from prophet import Prophet
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pmdarima as pm

from sklearn.metrics import mean_absolute_percentage_error
from sklearn.linear_model import Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor
import lightgbm as lgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials, space_eval

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.float_format', '{:,.2f}'.format)


## 3. Setting Up the Connector

In [ ]:
# load the credentials
with open('./credentials.json', 'r') as config_file:
    config = json.load(config_file)

snowflake_creds = config['snowflake_credentials']
ctx = snowflake.connector.connect(**snowflake_creds)


In [ ]:
cur = ctx.cursor()

## 4. Load Datasets

In [ ]:
cur.execute("""
SELECT * FROM IRONONETECH_DB.PUBLIC.V_FORECAST_CURVE_CONSOLIDATED;
""")
df = cur.fetch_pandas_all()
df.head()


## 5. Data Preparation

> **Buckeye change:** added `SUB_PORTFOLIO = 'PP'` filter vs Fingerhut notebook.

In [ ]:
portfolio       = 'Buckeye'
sub_portfolio   = 'PP'
last_input_date = '2024-12-31'

# ── Filter: portfolio + sub_portfolio + Actual only ──────────────────────
portfolio_df = df[df['PORTFOLIO'] == portfolio]
portfolio_df = portfolio_df[portfolio_df['SUB_PORTFOLIO'] == sub_portfolio]
portfolio_df = portfolio_df[portfolio_df['FORECAST_TYPE'] == 'Actual']

metrics = portfolio_df['METRIC'].unique()
print('Available metrics:', metrics)


In [ ]:
dff = portfolio_df.groupby(['DATE', 'METRIC']).sum().reset_index()
dff


In [ ]:
data_df = dff.pivot(
    index='DATE',
    columns='METRIC',
    values='METRIC_VALUE'
).sort_index()

# ── Train / Test split ───────────────────────────────────────────────────
train_df = data_df[data_df.index <= datetime.date.fromisoformat(last_input_date)]
test_df  = data_df[data_df.index >  datetime.date.fromisoformat(last_input_date)]

print(f'Training : {len(train_df)} samples | {train_df.index.min()} → {train_df.index.max()}')
print(f'Test     : {len(test_df)}  samples | {test_df.index.min()} → {test_df.index.max()}')
train_df.head()


## 6. Helper Functions

In [ ]:
# ── Plotting helper (identical to existing notebook) ─────────────────────
def plot_results(pred_df, metric):
    plt.figure(figsize=(20, 6))
    plt.title(metric)
    sns.lineplot(data=pred_df, x='DATE', y='METRIC_VALUE',
                 hue='FORECAST_TYPE', errorbar=None)
    plt.xlabel('DATE')
    plt.ylabel('METRIC_VALUE')
    plt.xticks(pred_df['DATE'].unique(), rotation=90)
    plt.tight_layout()
    plt.show()


# ── add_reforecast_data (adapted for Buckeye: SUB_PORTFOLIO filter) ───────
def add_reforecast_data(pred_df, main_df, portfolio, sub_portfolio,
                         metric, forecast_types):
    """
    Fetch existing reforecast rows (e.g. '2025 0+12') from main df
    and append to pred_df so they appear in the plot.
    """
    df_ref = main_df[
        (main_df['PORTFOLIO']     == portfolio) &
        (main_df['SUB_PORTFOLIO'] == sub_portfolio) &
        (main_df['DATE'].astype(str).isin(pred_df['DATE'].astype(str))) &
        (main_df['METRIC']        == metric) &
        (main_df['FORECAST_TYPE'].isin(forecast_types))
    ][['DATE', 'FORECAST_TYPE', 'METRIC_VALUE']].copy()

    df_ref['DATE'] = df_ref['DATE'].apply(
        lambda x: datetime.date.fromisoformat(str(x)))
    pred_df = pd.concat([pred_df, df_ref], axis=0, ignore_index=True)
    return pred_df


# ── Safe MAPE (skip NaN & zero actuals) ──────────────────────────────────
def safe_mape(y_true, y_pred):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask   = (~np.isnan(y_true)) & (~np.isnan(y_pred)) & (y_true != 0)
    if mask.sum() == 0:
        return np.nan
    return mean_absolute_percentage_error(y_true[mask], y_pred[mask]) * 100


# ── Walk-forward CV splits ────────────────────────────────────────────────
def ts_cv_splits(series, n_splits=3, val_size=2, min_train=10):
    n = len(series)
    for fold in range(n_splits):
        val_end   = n - (n_splits - 1 - fold) * val_size
        val_start = val_end - val_size
        if val_start < min_train:
            continue
        yield list(range(val_start)), list(range(val_start, val_end))


# ── Generic stat CV MAPE ──────────────────────────────────────────────────
def stat_cv_mape(predict_fn, series, n_splits=3, val_size=2, min_train=10):
    fold_mapes = []
    for tr_idx, val_idx in ts_cv_splits(series, n_splits, val_size, min_train):
        tr, val = series.iloc[tr_idx], series.iloc[val_idx]
        try:
            preds = predict_fn(tr, len(val))
            m     = safe_mape(val.values, np.array(preds))
            if not np.isnan(m):
                fold_mapes.append(m)
        except Exception:
            pass
    return np.mean(fold_mapes) if fold_mapes else np.nan


# ── ML walk-forward CV ────────────────────────────────────────────────────
def ml_cv_mape(model_factory, series, n_splits=3, val_size=2,
               min_train=10, use_scaler=False):
    fold_mapes = []
    for tr_idx, val_idx in ts_cv_splits(series, n_splits, val_size, min_train):
        tr_s, val_s = series.iloc[tr_idx], series.iloc[val_idx]
        X_tr, y_tr, _ = build_features(tr_s)
        if len(X_tr) < 4:
            continue
        scaler = StandardScaler() if use_scaler else None
        if scaler:
            X_tr = scaler.fit_transform(X_tr)
        try:
            mdl = model_factory()
            mdl.fit(X_tr, y_tr)
            val_dates = pd.date_range(
                start=pd.Timestamp(tr_s.index.max()) + pd.DateOffset(months=1),
                periods=len(val_s), freq='MS').date
            preds = recursive_ml_forecast(mdl, tr_s, val_dates, scaler=scaler)
            m     = safe_mape(val_s.values, np.array(preds))
            if not np.isnan(m):
                fold_mapes.append(m)
        except Exception:
            pass
    return np.mean(fold_mapes) if fold_mapes else np.nan


## 7. Target Metric & Test Configuration

In [ ]:
target_metric   = 'Beginning Outstanding Loans'
N_FORECAST      = 13   # Jan-2025 → Jan-2026

# ── Series ───────────────────────────────────────────────────────────────
train_series = train_df[target_metric].dropna()
test_series  = test_df[target_metric].dropna()

print(f'Train : {len(train_series)} pts | {train_series.index.min()} → {train_series.index.max()}')
print(f'Test  : {len(test_series)} pts | {test_series.index.min()} → {test_series.index.max()}')

# ── Forecast dates (these align with the test period) ────────────────────
last_train_date = pd.Timestamp(train_series.index.max())
forecast_dates  = pd.date_range(
    start=last_train_date + pd.DateOffset(months=1),
    periods=N_FORECAST, freq='MS'
).date
print(f'\nForecast dates: {forecast_dates[0]} → {forecast_dates[-1]}')

# ── Quick overview plot ───────────────────────────────────────────────────
plt.figure(figsize=(14, 4))
plt.plot(pd.to_datetime(train_series.index), train_series.values,
         'bo-', label='Actual (train)', linewidth=2, markersize=5)
plt.plot(pd.to_datetime(test_series.index), test_series.values,
         'r^--', label='Actual (test)', linewidth=2, markersize=7)
plt.axvline(pd.Timestamp(last_input_date), color='grey',
            linestyle=':', label='train/test split')
plt.title(f'{target_metric} — {portfolio}/{sub_portfolio}')
plt.xlabel('DATE'); plt.ylabel('VALUE')
plt.xticks(rotation=45); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Collect all results here ─────────────────────────────────────────────
results_mape = {}  # {model_name: {'cv_mape': float, 'test_mape': float}}

# ── Benchmarks from senior notebook ──────────────────────────────────────
BENCH_2025_0_12     = 0.67   # existing statistical reforecast MAPE
BENCH_SENIOR_PROPHET = 1.38  # senior Prophet MAPE


# ──────────────────────────────────────────────────────────────────────────
# CORE PLOT HELPER  →  reproduces Image-2 style exactly
# Shows: Actual (test) | model PRED | 2025 0+12  for the test window
# ──────────────────────────────────────────────────────────────────────────
def build_model_plot_df(model_name, preds_array,
                         test_series, forecast_dates,
                         main_df, reforecast_tag='2025 0+12'):
    """
    Build the melted DataFrame needed by plot_results().
    Includes: Actual (test period), model prediction, 2025 0+12 reforecast.
    preds_array  : 1-D array of length N_FORECAST, aligned to forecast_dates.
    """
    # 1. Actual rows (test period only — mirrors Image 2)
    actual_rows = pd.DataFrame({
        'DATE'         : list(test_series.index),
        'METRIC_VALUE' : list(test_series.values),
        'FORECAST_TYPE': 'Actual'
    })

    # 2. Model prediction rows
    pred_rows = pd.DataFrame({
        'DATE'         : list(forecast_dates),
        'METRIC_VALUE' : list(preds_array),
        'FORECAST_TYPE': model_name
    })

    # 3. Existing 2025 0+12 from main Snowflake df
    ref = main_df[
        (main_df['PORTFOLIO']     == portfolio) &
        (main_df['SUB_PORTFOLIO'] == sub_portfolio) &
        (main_df['METRIC']        == target_metric) &
        (main_df['FORECAST_TYPE'] == reforecast_tag)
    ][['DATE', 'METRIC_VALUE', 'FORECAST_TYPE']].copy()
    ref['DATE'] = ref['DATE'].apply(lambda x: datetime.date.fromisoformat(str(x)))
    # keep only dates that overlap with forecast window
    ref = ref[ref['DATE'].isin(forecast_dates)]

    plot_df = pd.concat([actual_rows, pred_rows, ref],
                         axis=0, ignore_index=True)
    return plot_df


# ──────────────────────────────────────────────────────────────────────────
# EVALUATE + PLOT  for one model
# ──────────────────────────────────────────────────────────────────────────
def evaluate_and_plot(model_name, preds_array, cv_mape):
    """
    1. Computes test MAPE against actual test_series.
    2. Stores in results_mape.
    3. Plots Image-2 style chart.
    """
    # Align predictions to test_series dates
    pred_series = pd.Series(preds_array,
                             index=pd.to_datetime(forecast_dates))
    test_idx    = pd.to_datetime(test_series.index)
    overlap     = pred_series.index.intersection(test_idx)

    if len(overlap) > 0:
        test_mape = safe_mape(
            test_series.loc[overlap.date].values,
            pred_series.loc[overlap].values
        )
    else:
        test_mape = np.nan

    results_mape[model_name] = {'cv_mape': cv_mape, 'test_mape': test_mape}

    print(f'{model_name:<25}  CV MAPE: {cv_mape:6.2f}%  |  Test MAPE: {test_mape:6.2f}%')

    # Build & plot
    plot_df = build_model_plot_df(model_name, preds_array,
                                   test_series, forecast_dates, df)
    plot_results(plot_df, f'{target_metric} — {model_name}')


## 8. Feature Engineering for ML Models

In [ ]:
def build_features(series):
    df_feat = pd.DataFrame({'y': series.values}, index=series.index)
    df_feat.index = pd.to_datetime(df_feat.index)
    df_feat['lag_1']          = df_feat['y'].shift(1)
    df_feat['lag_2']          = df_feat['y'].shift(2)
    df_feat['rolling_mean_3'] = df_feat['y'].shift(1).rolling(3).mean()
    df_feat['month_sin']      = np.sin(2 * np.pi * df_feat.index.month / 12)
    df_feat['month_cos']      = np.cos(2 * np.pi * df_feat.index.month / 12)
    df_feat['trend']          = np.arange(len(df_feat))
    df_feat = df_feat.dropna()
    feature_names = ['lag_1','lag_2','rolling_mean_3','month_sin','month_cos','trend']
    return df_feat[feature_names].values, df_feat['y'].values, feature_names


def recursive_ml_forecast(model, train_series, forecast_dates, scaler=None):
    history = list(train_series.values)
    preds   = []
    for fd in pd.to_datetime(forecast_dates):
        lag_1          = history[-1]
        lag_2          = history[-2] if len(history) >= 2 else history[-1]
        rolling_mean_3 = np.mean(history[-3:]) if len(history) >= 3 else np.mean(history)
        month_sin      = np.sin(2 * np.pi * fd.month / 12)
        month_cos      = np.cos(2 * np.pi * fd.month / 12)
        trend          = len(history)
        feat = np.array([[lag_1, lag_2, rolling_mean_3, month_sin, month_cos, trend]])
        if scaler is not None:
            feat = scaler.transform(feat)
        pred = max(model.predict(feat)[0], 0)
        preds.append(pred)
        history.append(pred)
    return preds


X_train_ml, y_train_ml, feat_names = build_features(train_series)
scaler_ml      = StandardScaler()
X_train_scaled = scaler_ml.fit_transform(X_train_ml)
print(f'ML feature matrix: {X_train_ml.shape}  |  features: {feat_names}')


## 9. Prophet — Bayesian Hyperparameter Tuning (Hyperopt)

Same search space as Fingerhut notebook. Walk-forward CV, 3 folds.


In [ ]:
prophet_space = {
    'changepoint_prior_scale': hp.loguniform('changepoint_prior_scale',
                                              np.log(0.001), np.log(10)),
    'seasonality_prior_scale': hp.loguniform('seasonality_prior_scale',
                                              np.log(0.01), np.log(5)),
    'changepoint_range'      : hp.uniform('changepoint_range', 0.70, 0.95),
    'n_changepoints'         : hp.quniform('n_changepoints', 5, 30, 1),
}


def prophet_cv_objective(params):
    fold_mapes = []
    for tr_idx, val_idx in ts_cv_splits(train_series, n_splits=3,
                                         val_size=2, min_train=10):
        tr  = train_series.iloc[tr_idx]
        val = train_series.iloc[val_idx]
        df_p = pd.DataFrame({'ds': pd.to_datetime(tr.index), 'y': tr.values})
        df_p['floor'] = 0
        df_p['cap']   = df_p['y'].max() * 1.2
        df_val = pd.DataFrame({
            'ds': pd.to_datetime(val.index),
            'floor': 0,
            'cap'  : df_p['y'].max() * 1.2
        })
        try:
            m = Prophet(
                growth='logistic',
                changepoint_prior_scale = params['changepoint_prior_scale'],
                seasonality_prior_scale = params['seasonality_prior_scale'],
                changepoint_range       = params['changepoint_range'],
                n_changepoints          = int(params['n_changepoints']),
                seasonality_mode='additive',
                weekly_seasonality=False, daily_seasonality=False,
                yearly_seasonality=False,
            )
            m.fit(df_p)
            fc = np.maximum(m.predict(df_val)['yhat'].values, 0)
            mape = safe_mape(val.values, fc)
            if not np.isnan(mape):
                fold_mapes.append(mape)
        except Exception:
            fold_mapes.append(200)
    return {'loss': np.mean(fold_mapes) if fold_mapes else 200,
            'status': STATUS_OK}


In [ ]:
print('Running Hyperopt for Prophet (50 trials)...')
prophet_trials      = Trials()
best_raw            = fmin(
    fn        = prophet_cv_objective,
    space     = prophet_space,
    algo      = tpe.suggest,
    max_evals = 50,
    trials    = prophet_trials,
    rstate    = np.random.default_rng(42),
    verbose   = False,
)
best_prophet_params = space_eval(prophet_space, best_raw)
best_prophet_params['n_changepoints'] = int(best_prophet_params['n_changepoints'])

prophet_cv_mape = min(prophet_trials.losses())
print(f'\nBest CV MAPE   : {prophet_cv_mape:.2f}%')
print(f'Senior baseline: {BENCH_SENIOR_PROPHET}%')
print(f'Target 2025 0+12: {BENCH_2025_0_12}%')
print('\nBest hyperparameters:')
for k, v in best_prophet_params.items():
    print(f'  {k:<30}: {v}')


In [ ]:
# ── Loss curve ───────────────────────────────────────────────────────────
trial_losses = [t['result']['loss'] for t in prophet_trials.trials]
best_so_far  = [min(trial_losses[:i+1]) for i in range(len(trial_losses))]
plt.figure(figsize=(12, 4))
plt.plot(trial_losses, alpha=0.4, label='Trial MAPE')
plt.plot(best_so_far, color='red', linewidth=2, label='Best so far')
plt.axhline(BENCH_2025_0_12,     color='green',  linestyle='--',
            label=f'2025 0+12 target ({BENCH_2025_0_12}%)')
plt.axhline(BENCH_SENIOR_PROPHET, color='orange', linestyle='--',
            label=f'Senior Prophet ({BENCH_SENIOR_PROPHET}%)')
plt.xlabel('Trial'); plt.ylabel('CV MAPE (%)')
plt.title('Hyperopt — Prophet optimisation (Buckeye/PP)')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Train on full training data with best params ─────────────────────────
df_p_full = pd.DataFrame({
    'ds': pd.to_datetime(train_series.index),
    'y' : train_series.values
})
cap_full = df_p_full['y'].max() * 1.2
df_p_full['floor'] = 0
df_p_full['cap']   = cap_full

prophet_final = Prophet(
    growth='logistic',
    changepoint_prior_scale = best_prophet_params['changepoint_prior_scale'],
    seasonality_prior_scale = best_prophet_params['seasonality_prior_scale'],
    changepoint_range       = best_prophet_params['changepoint_range'],
    n_changepoints          = best_prophet_params['n_changepoints'],
    seasonality_mode='additive',
    weekly_seasonality=False, daily_seasonality=False, yearly_seasonality=False,
)
prophet_final.fit(df_p_full)

future_df = pd.DataFrame({'ds': pd.to_datetime(forecast_dates),
                          'floor': 0, 'cap': cap_full})
prophet_preds = np.maximum(
    prophet_final.predict(future_df)['yhat'].values, 0)

evaluate_and_plot('Prophet_Tuned', prophet_preds, prophet_cv_mape)


## 10. Statistical Models

Each sub-section follows the same pattern:
1. Define `predict_fn(train_series, n_steps) → list`
2. Run walk-forward CV → CV MAPE
3. Forecast the full 13-month test window
4. Call `evaluate_and_plot()` → prints both MAPEs + Image-2 style chart


In [ ]:
def run_stat_model(name, predict_fn, series, n_forecast, forecast_dates):
    """Compute CV MAPE, produce 13-step forecast, evaluate & plot."""
    cv     = stat_cv_mape(predict_fn, series)
    preds  = np.array(predict_fn(series, n_forecast))
    evaluate_and_plot(name, preds, cv)
    return preds


In [ ]:
# 10.1  Naive — last value carried forward
def naive_predict(series, n_steps):
    return [series.iloc[-1]] * n_steps

naive_preds = run_stat_model('Naive', naive_predict,
                              train_series, N_FORECAST, forecast_dates)


In [ ]:
# 10.2  Moving Average  MA(k)  k = 2, 3, 6
ma_results = {}
for k in [2, 3, 6]:
    def _ma(series, n_steps, w=k):
        hist  = list(series.values)
        preds = []
        for _ in range(n_steps):
            p = np.mean(hist[-w:])
            preds.append(p)
            hist.append(p)
        return preds
    ma_results[k] = run_stat_model(f'MA({k})', _ma,
                                    train_series, N_FORECAST, forecast_dates)


In [ ]:
# 10.3  Simple Exponential Smoothing (optimised alpha)
def ses_predict(series, n_steps):
    fit = SimpleExpSmoothing(series.values,
                              initialization_method='estimated').fit(optimized=True)
    return fit.forecast(n_steps).tolist()

ses_preds = run_stat_model('SES', ses_predict,
                            train_series, N_FORECAST, forecast_dates)


In [ ]:
# 10.4  Holt (Double Exponential Smoothing — additive trend)
def holt_predict(series, n_steps):
    fit = ExponentialSmoothing(
        series.values, trend='add', seasonal=None,
        initialization_method='estimated').fit(optimized=True)
    return fit.forecast(n_steps).tolist()

holt_preds = run_stat_model('Holt', holt_predict,
                             train_series, N_FORECAST, forecast_dates)


In [ ]:
# 10.5  Holt-Winters  (requires >= 2 seasonal cycles = 24 pts)
if len(train_series) >= 24:
    def hw_predict(series, n_steps):
        fit = ExponentialSmoothing(
            series.values, trend='add', seasonal='add',
            seasonal_periods=12,
            initialization_method='estimated').fit(optimized=True)
        return fit.forecast(n_steps).tolist()
    hw_preds = run_stat_model('Holt-Winters', hw_predict,
                               train_series, N_FORECAST, forecast_dates)
else:
    print(f'Holt-Winters skipped — {len(train_series)} pts (need ≥ 24)')


In [ ]:
# 10.6  AutoRegression  AR(p)  p = 1, 2
for p in [1, 2]:
    def _ar(series, n_steps, lags=p):
        fit   = AutoReg(series.values, lags=lags, old_names=False).fit()
        hist  = list(series.values)
        coef  = fit.params
        preds = []
        for _ in range(n_steps):
            val = coef[0] + sum(coef[i] * hist[-i] for i in range(1, lags+1))
            preds.append(val)
            hist.append(val)
        return preds
    run_stat_model(f'AR({p})', _ar, train_series, N_FORECAST, forecast_dates)


In [ ]:
# 10.7  ARIMA — common monthly orders
for order in [(1,1,0), (0,1,1), (1,1,1), (2,1,0), (0,1,2)]:
    def _arima(series, n_steps, o=order):
        return ARIMA(series.values, order=o).fit().forecast(n_steps).tolist()
    run_stat_model(f'ARIMA{order}', _arima,
                   train_series, N_FORECAST, forecast_dates)


In [ ]:
# 10.8  Auto-ARIMA (pmdarima stepwise)
def auto_arima_predict(series, n_steps):
    model = pm.auto_arima(
        series.values, seasonal=False, stepwise=True,
        information_criterion='aic', error_action='ignore',
        suppress_warnings=True, max_p=3, max_q=3, max_d=2
    )
    return model.predict(n_periods=n_steps).tolist()

auto_arima_preds = run_stat_model('Auto-ARIMA', auto_arima_predict,
                                   train_series, N_FORECAST, forecast_dates)


In [ ]:
# 10.9  SARIMA(1,1,0)(1,0,0,12) — only if >= 24 training points
if len(train_series) >= 24:
    def sarima_predict(series, n_steps):
        fit = SARIMAX(series.values, order=(1,1,0),
                      seasonal_order=(1,0,0,12),
                      enforce_stationarity=False,
                      enforce_invertibility=False).fit(disp=False)
        return fit.forecast(n_steps).tolist()
    run_stat_model('SARIMA(1,1,0)(1,0,0,12)', sarima_predict,
                   train_series, N_FORECAST, forecast_dates)
else:
    print(f'SARIMA skipped — {len(train_series)} pts (need ≥ 24)')


## 11. ML Models

All models use standardised features. Regularisation is heavy to guard against
overfitting on the small training set.


In [ ]:
# 11.1  Ridge Regression (alpha via RidgeCV)
ridge_cv_model = RidgeCV(alphas=[0.01, 0.1, 1, 10, 100, 500], cv=3)
ridge_cv_model.fit(X_train_scaled, y_train_ml)
best_ridge_alpha = ridge_cv_model.alpha_
print(f'Ridge best alpha: {best_ridge_alpha}')

ridge_cv_mape = ml_cv_mape(lambda: Ridge(alpha=best_ridge_alpha),
                            train_series, use_scaler=True)
ridge_preds   = recursive_ml_forecast(ridge_cv_model, train_series,
                                       forecast_dates, scaler=scaler_ml)
evaluate_and_plot('Ridge', np.array(ridge_preds), ridge_cv_mape)


In [ ]:
# 11.2  Lasso Regression (alpha via LassoCV)
lasso_cv_model = LassoCV(cv=3, max_iter=5000)
lasso_cv_model.fit(X_train_scaled, y_train_ml)
best_lasso_alpha = lasso_cv_model.alpha_
print(f'Lasso best alpha: {best_lasso_alpha:.6f}')

lasso_cv_mape = ml_cv_mape(lambda: Lasso(alpha=best_lasso_alpha, max_iter=5000),
                            train_series, use_scaler=True)
lasso_preds   = recursive_ml_forecast(lasso_cv_model, train_series,
                                       forecast_dates, scaler=scaler_ml)
evaluate_and_plot('Lasso', np.array(lasso_preds), lasso_cv_mape)


In [ ]:
# 11.3  ElasticNet
enet_cv_model = ElasticNetCV(l1_ratio=[0.2, 0.5, 0.8], cv=3, max_iter=5000)
enet_cv_model.fit(X_train_scaled, y_train_ml)
print(f'ElasticNet alpha={enet_cv_model.alpha_:.6f}, l1_ratio={enet_cv_model.l1_ratio_}')

enet_cv_mape = ml_cv_mape(
    lambda: ElasticNet(alpha=enet_cv_model.alpha_,
                       l1_ratio=enet_cv_model.l1_ratio_, max_iter=5000),
    train_series, use_scaler=True)
enet_preds   = recursive_ml_forecast(enet_cv_model, train_series,
                                      forecast_dates, scaler=scaler_ml)
evaluate_and_plot('ElasticNet', np.array(enet_preds), enet_cv_mape)


In [ ]:
# 11.4  Support Vector Regression (RBF)
svr_model   = SVR(kernel='rbf', C=1.0, epsilon=0.05)
svr_model.fit(X_train_scaled, y_train_ml)

svr_cv_mape = ml_cv_mape(lambda: SVR(kernel='rbf', C=1.0, epsilon=0.05),
                          train_series, use_scaler=True)
svr_preds   = recursive_ml_forecast(svr_model, train_series,
                                     forecast_dates, scaler=scaler_ml)
evaluate_and_plot('SVR_RBF', np.array(svr_preds), svr_cv_mape)


In [ ]:
# 11.5  XGBoost (shallow, heavily regularised)
xgb_model = XGBRegressor(
    n_estimators=50, max_depth=2, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=5.0,
    random_state=42, verbosity=0
)
xgb_model.fit(X_train_scaled, y_train_ml)

xgb_cv_mape = ml_cv_mape(
    lambda: XGBRegressor(n_estimators=50, max_depth=2, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         reg_alpha=1.0, reg_lambda=5.0,
                         random_state=42, verbosity=0),
    train_series, use_scaler=True)
xgb_preds   = recursive_ml_forecast(xgb_model, train_series,
                                     forecast_dates, scaler=scaler_ml)
evaluate_and_plot('XGBoost', np.array(xgb_preds), xgb_cv_mape)


In [ ]:
# 11.6  LightGBM (small memory, fast)
lgb_model = lgb.LGBMRegressor(
    n_estimators=50, num_leaves=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=5.0,
    random_state=42, verbose=-1
)
lgb_model.fit(X_train_scaled, y_train_ml)

lgb_cv_mape = ml_cv_mape(
    lambda: lgb.LGBMRegressor(n_estimators=50, num_leaves=4,
                               learning_rate=0.05, subsample=0.8,
                               colsample_bytree=0.8, reg_alpha=1.0,
                               reg_lambda=5.0, random_state=42, verbose=-1),
    train_series, use_scaler=True)
lgb_preds   = recursive_ml_forecast(lgb_model, train_series,
                                     forecast_dates, scaler=scaler_ml)
evaluate_and_plot('LightGBM', np.array(lgb_preds), lgb_cv_mape)


In [ ]:
# ── Feature importances (XGBoost & LightGBM) ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
pd.Series(xgb_model.feature_importances_, index=feat_names) \
    .sort_values().plot(kind='barh', ax=axes[0], title='XGBoost Feature Importance')
pd.Series(lgb_model.feature_importances_, index=feat_names) \
    .sort_values().plot(kind='barh', ax=axes[1], title='LightGBM Feature Importance')
plt.tight_layout(); plt.show()


## 12. Final Results Comparison

In [ ]:
# ── Summary table ────────────────────────────────────────────────────────
rows = []
for name, vals in results_mape.items():
    rows.append({
        'Model'        : name,
        'CV MAPE (%)'  : round(vals['cv_mape'],   2) if not np.isnan(vals['cv_mape'])   else np.nan,
        'Test MAPE (%)': round(vals['test_mape'],  2) if not np.isnan(vals['test_mape']) else np.nan,
        'Type': 'ML'         if name in ['Ridge','Lasso','ElasticNet',
                                         'SVR_RBF','XGBoost','LightGBM']
               else 'Prophet' if 'Prophet' in name
               else 'Statistical'
    })

results_df = pd.DataFrame(rows).sort_values('Test MAPE (%)')

print('\n' + '='*72)
print(f"{'Model':<30} {'CV MAPE':>10}  {'Test MAPE':>10}  {'Type':<14}")
print('='*72)
for _, r in results_df.iterrows():
    beat = ' ★' if r['Test MAPE (%)'] < BENCH_2025_0_12 else ''
    print(f"{r['Model']:<30} {r['CV MAPE (%)']:>10.2f}  "
          f"{r['Test MAPE (%)']:>10.2f}  {r['Type']:<14}{beat}")
print('-'*72)
print(f"{'2025 0+12  [TARGET]':<30} {'—':>10}  {BENCH_2025_0_12:>10.2f}  {'Benchmark':<14}")
print(f"{'Senior Prophet [BASELINE]':<30} {'—':>10}  {BENCH_SENIOR_PROPHET:>10.2f}  {'Benchmark':<14}")
print('='*72)
print('★ = beats 2025 0+12 target (0.67%)')

results_df


In [ ]:
# ── Dual bar chart: CV MAPE vs Test MAPE ────────────────────────────────
plot_r = results_df.dropna(subset=['Test MAPE (%)']).copy()
x      = np.arange(len(plot_r))
width  = 0.38

type_colors = {'Prophet': '#4C72B0', 'Statistical': '#55A868', 'ML': '#DD8452'}
bar_c = [type_colors.get(t, 'grey') for t in plot_r['Type']]

fig, ax = plt.subplots(figsize=(max(14, len(plot_r)*0.8), 6))
ax.bar(x - width/2, plot_r['CV MAPE (%)'],   width, color=bar_c,
       alpha=0.55, edgecolor='white', label='CV MAPE')
ax.bar(x + width/2, plot_r['Test MAPE (%)'], width, color=bar_c,
       alpha=0.95, edgecolor='white', label='Test MAPE')

ax.axhline(BENCH_2025_0_12,     color='green',  linestyle='--', linewidth=1.8,
           label=f'Target 2025 0+12 ({BENCH_2025_0_12}%)')
ax.axhline(BENCH_SENIOR_PROPHET, color='orange', linestyle='--', linewidth=1.8,
           label=f'Senior Prophet ({BENCH_SENIOR_PROPHET}%)')

ax.set_xticks(x)
ax.set_xticklabels(plot_r['Model'], rotation=45, ha='right')
ax.set_ylabel('MAPE (%)')
ax.set_title(f'{target_metric} — All Models: CV vs Test MAPE ({portfolio}/{sub_portfolio})')

from matplotlib.patches import Patch
legend_el = [
    Patch(facecolor='#4C72B0', alpha=0.8, label='Prophet'),
    Patch(facecolor='#55A868', alpha=0.8, label='Statistical'),
    Patch(facecolor='#DD8452', alpha=0.8, label='ML'),
]
ax.legend(handles=legend_el + [
    plt.Line2D([0],[0], color='green',  linestyle='--',
               label=f'Target ({BENCH_2025_0_12}%)'),
    plt.Line2D([0],[0], color='orange', linestyle='--',
               label=f'Senior Prophet ({BENCH_SENIOR_PROPHET}%)'),
])
plt.tight_layout()
plt.show()


In [ ]:
# ── Top-5 models overlaid on one Image-2 style chart ────────────────────
top5_names = results_df.dropna(subset=['Test MAPE (%)']).head(5)['Model'].tolist()

# Rebuild predictions dict (all models that were run)
_pred_map = {
    'Prophet_Tuned'              : prophet_preds,
    'Naive'                      : naive_preds,
    'MA(2)'                      : ma_results.get(2, []),
    'MA(3)'                      : ma_results.get(3, []),
    'MA(6)'                      : ma_results.get(6, []),
    'SES'                        : ses_preds,
    'Holt'                       : holt_preds,
    'Auto-ARIMA'                 : auto_arima_preds,
    'Ridge'                      : ridge_preds,
    'Lasso'                      : lasso_preds,
    'ElasticNet'                 : enet_preds,
    'SVR_RBF'                    : svr_preds,
    'XGBoost'                    : xgb_preds,
    'LightGBM'                   : lgb_preds,
}

# Fetch 2025 0+12 reference once
ref_2025 = df[
    (df['PORTFOLIO']     == portfolio) &
    (df['SUB_PORTFOLIO'] == sub_portfolio) &
    (df['METRIC']        == target_metric) &
    (df['FORECAST_TYPE'] == '2025 0+12')
][['DATE', 'METRIC_VALUE']].copy()
ref_2025['DATE'] = ref_2025['DATE'].apply(lambda x: datetime.date.fromisoformat(str(x)))
ref_2025 = ref_2025[ref_2025['DATE'].isin(forecast_dates)]

fig, ax = plt.subplots(figsize=(20, 7))

# Actual test data
ax.plot(pd.to_datetime(test_series.index), test_series.values,
        'ko-', linewidth=2.5, markersize=6, label='Actual', zorder=10)

# 2025 0+12 benchmark
ax.plot(pd.to_datetime(ref_2025['DATE']), ref_2025['METRIC_VALUE'],
        color='green', linewidth=2, linestyle='--',
        label=f'2025 0+12 (target {BENCH_2025_0_12}%)', zorder=9)

# Top-5 model forecasts
cmap = plt.cm.tab10
for i, mname in enumerate(top5_names):
    preds = _pred_map.get(mname)
    if preds is None:
        continue
    tmape = results_mape[mname]['test_mape']
    ax.plot(pd.to_datetime(forecast_dates), preds,
            color=cmap(i), linewidth=1.8, linestyle='--', marker='o',
            markersize=4, alpha=0.85,
            label=f'{mname} ({tmape:.2f}%)')

ax.set_title(f'{target_metric} — Top-5 Models vs Actuals ({portfolio}/{sub_portfolio})',
             fontsize=13)
ax.set_xlabel('DATE')
ax.set_ylabel('METRIC_VALUE')
ax.legend(loc='best', fontsize=9)
plt.xticks(pd.to_datetime(forecast_dates), rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
# ── Best model summary ───────────────────────────────────────────────────
best_row   = results_df.dropna(subset=['Test MAPE (%)']).iloc[0]
best_name  = best_row['Model']
best_test  = best_row['Test MAPE (%)']
best_cv    = best_row['CV MAPE (%)']

print(f'Portfolio      : {portfolio} / {sub_portfolio}')
print(f'Best model     : {best_name}')
print(f'Test MAPE      : {best_test:.2f}%')
print(f'Target (0+12)  : {BENCH_2025_0_12}%')
print(f'Baseline        : {BENCH_SENIOR_PROPHET}%')
print(f'Improvement vs Senior Prophet : {BENCH_SENIOR_PROPHET - best_test:.2f} pp')
beat_target = 'YES ✓' if best_test < BENCH_2025_0_12 else 'NO — needs more tuning'
print(f'Beats 2025 0+12 target        : {beat_target}')
